In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/churn.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 0
N_FOLDS = 5


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))
data.head()


In [ ]:
data.info()


In [ ]:
# Быстрый EDA: churn по contract
plt.figure(figsize=(8, 4))
sns.barplot(data=data, x='contract_type', y='churn', palette='viridis', hue='contract_type', legend=False, errorbar=None)
plt.title('Churn rate по contract_type', fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
print("churn rate:", round(data["churn"].mean(), 3))


In [ ]:
# ======================
# 2. ПРИЗНАКИ
# ======================
num_cols = ["tenure_months", "monthly_charges", "support_tickets", "total_charges"]
cat_cols = ["contract_type", "internet_service"]

X = data[num_cols + cat_cols]
y = data["churn"]


In [ ]:
# ======================
# 3. ПАЙПЛАЙН
# ======================
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

pipe = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000)),
])


In [ ]:
# ======================
# 4. ОДИН SPLIT (для сравнения)
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y
)
pipe.fit(X_train, y_train)
single_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
print("single split AUC:", round(single_auc, 3))


In [ ]:
# ======================
# 5. STRATIFIED K-FOLD CV
# ======================
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
print("fold scores:", scores.round(3))
print("mean:", round(scores.mean(), 3), "std:", round(scores.std(), 3))


In [ ]:
# ======================
# 6. CV ДЛЯ ДВУХ МОДЕЛЕЙ
# ======================
rf_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])),
    ("clf", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_SEED)),
])

rf_scores = cross_val_score(rf_pipe, X, y, cv=cv, scoring="roc_auc")
print("RF fold scores:", rf_scores.round(3))
print("RF mean:", round(rf_scores.mean(), 3), "std:", round(rf_scores.std(), 3))


In [ ]:
# ======================
# 7. ВИЗУАЛИЗАЦИЯ РАЗБРОСА ПО ФОЛДАМ
# ======================
plot_df = pd.DataFrame({
    "fold": list(range(1, N_FOLDS + 1)) * 2,
    "auc": list(scores) + list(rf_scores),
    "model": ["LogReg"] * N_FOLDS + ["RandomForest"] * N_FOLDS,
})

plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df, x="fold", y="auc", hue="model", palette="viridis")
plt.axhline(scores.mean(), color="#440154", linestyle="--", lw=1, label="LogReg mean")
plt.axhline(rf_scores.mean(), color="#35b779", linestyle="--", lw=1, label="RF mean")
plt.title("ROC-AUC по фолдам (StratifiedKFold)", fontsize=14, pad=15)
plt.xlabel("Фолд", fontsize=12)
plt.ylabel("ROC-AUC", fontsize=12)
plt.ylim(min(scores.min(), rf_scores.min()) - 0.02, 1.0)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig(FIG / "churn_cv_folds.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Boxplot разброса метрик
plt.figure(figsize=(7, 5))
sns.boxplot(data=plot_df, x="model", y="auc", palette="viridis", hue="model", legend=False)
sns.stripplot(data=plot_df, x="model", y="auc", color="black", size=8, alpha=0.7)
plt.title("Разброс AUC между фолдами", fontsize=14, pad=15)
plt.ylabel("ROC-AUC", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "churn_cv_boxplot.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print("Один split = одна точка. CV показывает разброс метрики.")
print("Для классификации с дисбалансом — StratifiedKFold.")
